# Снятие фикстур: несколько индикаторов

Вставь ссылки на страницы индикаторов fedstat.ru в `URLS`, запусти все ячейки (нужен доступ к fedstat.ru). Для каждого индикатора сохраняются в `fixtures/`:
- `{id}_indicator.html` — страница (для парсера фильтров/CSRF);
- `{id}_data_ids.json` — разобранные поля-фильтры;
- `{id}_data.sdmx.xml` — сырой ответ SDMX (полный срез).

Потом просто пришли/сохрани — я прочту эти файлы и доведу парсер под разные структуры.

## 1. Ссылки на индикаторы

In [1]:
# вставь свои ссылки (или просто id-числа)
URLS = [
    "https://www.fedstat.ru/indicator/31074",
    # "https://www.fedstat.ru/indicator/XXXXX",
    # "https://www.fedstat.ru/indicator/YYYYY",
]
URLS = ["https://www.fedstat.ru/indicator/59345"]
OUT = "fixtures"

## 2. Скачать и сохранить (сырой HTML + SDMX)

In [3]:
import os, sys, re, json

# гарантируем сохранение в КОРНЕВОЙ fixtures/, даже если ноутбук запущен из notebooks/
here = os.getcwd()
root = here if os.path.exists(os.path.join(here, 'fedstat')) else os.path.dirname(here)
if root not in sys.path:
    sys.path.insert(0, root)
os.chdir(root)

from fedstat.client import FedstatClient
from fedstat.discovery import parse_indicator_page, build_download_body
from fedstat.filters import select_rows

os.makedirs(OUT, exist_ok=True)
print('сохраняю в:', os.path.abspath(OUT))
client = FedstatClient(retry_max_times=6)

def ind_id(u):
    m = re.search(r'/indicator/(\d+)', str(u)) or re.search(r'(\d+)', str(u))
    return m.group(1)

for url in URLS:
    iid = ind_id(url)
    print(f'\n=== {iid} ({url}) ===')
    try:
        html = client.get_indicator_html(iid)
        with open(f'{OUT}/{iid}_indicator.html', 'w', encoding='utf-8') as fh:
            fh.write(html)
        di = parse_indicator_page(html, iid)
        with open(f'{OUT}/{iid}_data_ids.json', 'w', encoding='utf-8') as fh:
            json.dump(di.rows, fh, ensure_ascii=False, indent=2)
        print(f'  HTML+фильтры ок | полей: {len(di.fields())} | CSRF: {bool(di.csrf_token)}')
        body = build_download_body(di, select_rows(di, None))
        raw = client.download(body, data_format='sdmx', referer=f'{client.base_url}/indicator/{iid}')
        with open(f'{OUT}/{iid}_data.sdmx.xml', 'wb') as fh:
            fh.write(raw)
        print(f'  SDMX сохранён: {len(raw):,} байт')
        client.reset_session()
    except Exception as e:
        print(f'  [!] {type(e).__name__}: {str(e)[:180]}')
        client.reset_session()



=== 59345 (https://www.fedstat.ru/indicator/59345) ===
  HTML+фильтры ок | полей: 7 | CSRF: True
  SDMX сохранён: 64,676,059 байт


## 3. Что получилось (проверка парсинга)

In [ ]:
import glob, os
from fedstat import sdmx_to_dataframe

for p in sorted(glob.glob(f"{OUT}/*_data.sdmx.xml")):
    try:
        df = sdmx_to_dataframe(p)
        print(os.path.basename(p), f"{os.path.getsize(p):,} байт ->", df.shape)
        print("   столбцы:", list(df.columns))
    except Exception as e:
        print(os.path.basename(p), "ОШИБКА:", type(e).__name__, str(e)[:150])


---
Если по какому-то индикатору **503** — fedstat перегружен, перезапусти ячейку 2 позже. Если **302** на полном срезе — редкость (значит у индикатора особая структура), тоже сообщи. HTML сохраняется в любом случае — по нему уже можно чинить парсер.